# File Artifact Test Suite
More than 25 cells that each write a real file into the kernel's current working directory, using only commonly-available libraries. After running everything, open the workspace file tree and click into each generated file to test its preview: Excel viewer, PDF viewer, video player, and image preview.

Every file-writing cell only uses a relative filename, so it lands wherever this kernel's working directory is - the first cell prints that location.

In [ ]:
import os

print("Working directory:", os.getcwd())
print("Files here before this run:", sorted(os.listdir(".")))

Working directory: /tmp/2
Files here before this run: ['.cache', '.config', '.ipynb_checkpoints', '.ipython', '.jupyter', '.local', '.npm', 'DEPLOYMENT_STEPS.txt', 'data_ask_20260818_091022 (1).pdf', 'india_states.geojson', 'solix-mark-on-light.png']


## 1. Standalone image files (PNG / JPG)

In [6]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 4 * np.pi, 200)
plt.figure(figsize=(6, 3))
plt.plot(x, np.sin(x), label="sin(x)")
plt.plot(x, np.cos(x), label="cos(x)")
plt.title("Line chart - saved as a file")
plt.legend()
plt.savefig("chart_line.png", dpi=120)
plt.close()
print("Wrote chart_line.png:", os.path.getsize("chart_line.png"), "bytes")

Wrote chart_line.png: 43735 bytes


In [7]:
categories = ["A", "B", "C", "D", "E"]
values = [23, 45, 12, 38, 30]
plt.figure(figsize=(6, 3))
plt.bar(categories, values, color="teal")
plt.title("Bar chart - saved as a file")
plt.savefig("chart_bar.png", dpi=120)
plt.close()
print("Wrote chart_bar.png:", os.path.getsize("chart_bar.png"), "bytes")

Wrote chart_bar.png: 11512 bytes


In [ ]:
from PIL import Image

width, height = 300, 200
gradient = Image.new("RGB", (width, height))
for py in range(height):
    for px in range(width):
        gradient.putpixel((px, py), (int(255 * px / width), int(255 * py / height), 180))
gradient.save("gradient.jpg", quality=90)
print("Wrote gradient.jpg:", os.path.getsize("gradient.jpg"), "bytes")

Wrote gradient.jpg: 4989 bytes


## 2. Graphs as a multi-page PDF

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

rng = np.random.default_rng(7)
with PdfPages("charts_report.pdf") as pdf:
    fig1, ax1 = plt.subplots(figsize=(6, 4))
    ax1.plot(x, np.sin(x))
    ax1.set_title("Page 1 - line chart")
    pdf.savefig(fig1)
    plt.close(fig1)

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.bar(categories, values, color="orange")
    ax2.set_title("Page 2 - bar chart")
    pdf.savefig(fig2)
    plt.close(fig2)

    fig3, ax3 = plt.subplots(figsize=(6, 4))
    ax3.scatter(rng.normal(size=150), rng.normal(size=150), alpha=0.6)
    ax3.set_title("Page 3 - scatter chart")
    pdf.savefig(fig3)
    plt.close(fig3)

print("Wrote charts_report.pdf:", os.path.getsize("charts_report.pdf"), "bytes")

Wrote charts_report.pdf: 19185 bytes


In [10]:
assert os.path.exists("charts_report.pdf") and os.path.getsize("charts_report.pdf") > 0
print("PASS: charts_report.pdf exists and is non-empty (3 pages written)")

PASS: charts_report.pdf exists and is non-empty (3 pages written)


## 3. Excel workbooks (.xlsx)

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "id": range(1, 11),
    "name": [f"Item_{i}" for i in range(1, 11)],
    "amount": rng.uniform(10, 1000, size=10).round(2),
})
df.to_excel("report_data.xlsx", index=False, sheet_name="Report")
print("Wrote report_data.xlsx:", os.path.getsize("report_data.xlsx"), "bytes")

In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill

wb = openpyxl.Workbook()
ws1 = wb.active
ws1.title = "Summary"
ws1.append(["Metric", "Value"])
for cell in ws1[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill(start_color="0747A6", end_color="0747A6", fill_type="solid")
ws1.append(["Total records", 10])
ws1.append(["Average amount", round(df["amount"].mean(), 2)])

ws2 = wb.create_sheet("Detail")
ws2.append(list(df.columns))
for row in df.itertuples(index=False):
    ws2.append(list(row))

wb.save("multi_sheet_report.xlsx")
print("Wrote multi_sheet_report.xlsx with sheets:", wb.sheetnames)

In [ ]:
roundtrip = pd.read_excel("report_data.xlsx", sheet_name="Report")
assert len(roundtrip) == len(df)
print("PASS: report_data.xlsx round-trips through pandas.read_excel correctly")
roundtrip.head()

## 4. JSON files

In [ ]:
import json

config = {
    "app": "solix-notebook",
    "version": 1,
    "features": ["source-target", "schema-browser", "git"],
    "limits": {"max_schemas": 10, "max_tables": 10},
}
with open("config_test.json", "w") as f:
    json.dump(config, f, indent=2)
print("Wrote config_test.json:", os.path.getsize("config_test.json"), "bytes")

In [ ]:
with open("config_test.json") as f:
    loaded = json.load(f)
assert loaded == config
print("PASS: config_test.json round-trips correctly")

In [ ]:
unicode_payload = {"greeting": "caf\u00e9 \u4e2d\u6587 \U0001F680", "numbers": [1, 2, 3]}
with open("unicode_test.json", "w", encoding="utf-8") as f:
    json.dump(unicode_payload, f, indent=2, ensure_ascii=False)
print("Wrote unicode_test.json:", os.path.getsize("unicode_test.json"), "bytes")

## 5. Video file (with graceful fallback)

In [ ]:
video_path = None
try:
    import cv2

    frames_dir_writer = cv2.VideoWriter(
        "demo_video.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 10, (320, 240)
    )
    for frame_i in range(40):
        frame = np.zeros((240, 320, 3), dtype=np.uint8)
        cx = int(20 + (280) * (frame_i / 39))
        cv2.circle(frame, (cx, 120), 20, (0, 200, 255), -1)
        frames_dir_writer.write(frame)
    frames_dir_writer.release()
    video_path = "demo_video.mp4"
    print("Wrote demo_video.mp4 using OpenCV")
except Exception as e:
    print("OpenCV unavailable or failed (", e, ") - falling back to an animated GIF")

In [ ]:
if video_path is None:
    try:
        from PIL import Image, ImageDraw

        frames = []
        for frame_i in range(40):
            img = Image.new("RGB", (320, 240), (20, 20, 30))
            draw = ImageDraw.Draw(img)
            cx = int(20 + 280 * (frame_i / 39))
            draw.ellipse([cx - 20, 100, cx + 20, 140], fill=(0, 200, 255))
            frames.append(img)
        frames[0].save(
            "demo_video.gif", save_all=True, append_images=frames[1:], duration=80, loop=0
        )
        video_path = "demo_video.gif"
        print("Wrote demo_video.gif using Pillow (video fallback)")
    except Exception as e:
        print("Pillow GIF fallback also failed:", e)

print("Final video artifact:", video_path)

In [ ]:
assert video_path is not None and os.path.exists(video_path)
print(f"PASS: {video_path} exists,", os.path.getsize(video_path), "bytes")

## 6. Round-trip / misc checks

In [ ]:
import csv

with open("sample_data.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "label"])
    for i in range(5):
        writer.writerow([i, f"label_{i}"])

with open("sample_data.csv") as f:
    rows = list(csv.reader(f))
assert len(rows) == 6  # header + 5 data rows
print("PASS: sample_data.csv written and read back correctly")

PASS: sample_data.csv written and read back correctly


In [ ]:
import zipfile

artifact_names = [
    "chart_line.png", "chart_bar.png", "gradient.jpg", "charts_report.pdf",
    "report_data.xlsx", "multi_sheet_report.xlsx", "config_test.json",
    "unicode_test.json", video_path, "sample_data.csv",
]
with zipfile.ZipFile("test_artifacts_bundle.zip", "w") as zf:
    for name in artifact_names:
        if name and os.path.exists(name):
            zf.write(name)

with zipfile.ZipFile("test_artifacts_bundle.zip") as zf:
    print("Zip contents:", zf.namelist())

In [ ]:
try:
    open("this_file_does_not_exist.xyz").read()
except FileNotFoundError as e:
    print("Caught expected error opening a missing file:", e)
    print("PASS: missing-file error handling works")

In [16]:
expected_files = [
    "chart_line.png", "chart_bar.png", "gradient.jpg", "charts_report.pdf",
    "report_data.xlsx", "multi_sheet_report.xlsx", "config_test.json",
    "unicode_test.json", video_path, "sample_data.csv", "test_artifacts_bundle.zip",
]
print("File artifact checklist:")
all_ok = True
for name in expected_files:
    ok = bool(name) and os.path.exists(name)
    all_ok = all_ok and ok
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
print()
print("ALL PASS" if all_ok else "SOME CHECKS FAILED")

Error: 

## Done
Open the workspace file tree now and click into `chart_line.png`, `gradient.jpg`, `charts_report.pdf`, `report_data.xlsx`, `multi_sheet_report.xlsx`, and the video artifact (`demo_video.mp4` or `demo_video.gif`) to exercise each file-preview viewer.